<a href="https://colab.research.google.com/github/pranatixsharma/Masculine_defaults_Indian_youtube/blob/main/get_LDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q indic-nlp-library stopwordsiso nltk scikit-learn pandas
import nltk
nltk.download('stopwords', quiet=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 7.1 MB/s eta 0:00:00


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Map each community to its transcript folder
COMMUNITY_FOLDERS = {
    "business_finance": "/content/drive/MyDrive/Transcripts_CSS/transcripts_business",
}

OUTPUT_DIR  = "/content/drive/MyDrive/lda_output/topics_business"
N_TOPICS    = 20
N_TOP_WORDS = 50
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
import glob, os

for community, folder in COMMUNITY_FOLDERS.items():
    files = glob.glob(os.path.join(folder, "*.json"))
    print(f"{community}: {len(files)} files found at {folder}")

business_finance: 1136 files found at /content/drive/MyDrive/Transcripts_CSS/transcripts_business


Build Stopword list

In [ ]:
from nltk.corpus import stopwords as nltk_stopwords
from stopwordsiso import stopwords as iso_stopwords

english_stops = set(nltk_stopwords.words('english'))
hindi_stops   = set(iso_stopwords("hi"))

custom_stops = {
    # Hindi fillers common in YouTube speech
    "है", "हैं", "और", "के", "का", "की", "को", "में", "से", "पर",
    "कि", "यह", "वह", "इस", "उस", "एक", "भी", "तो", "हो", "ने",
    "जो", "कर", "हम", "आप", "था", "थे", "थी", "जा", "रहा", "रही",
    "वो", "ये", "उन", "इन", "जब", "तब", "अब", "यहाँ", "वहाँ",
    # English fillers common in Hinglish
    "like", "okay", "ok", "right", "know", "see", "now", "so",
    "actually", "basically", "also", "just", "very", "really",
    "going", "come", "get", "let", "say", "said", "one", "two",
    # Channel/video meta words
    "video", "channel", "subscribe", "comment", "watch", "today",
    "guys", "hello", "hi", "bye", "thanks", "share", "like",
}

ALL_STOPS = english_stops | hindi_stops | custom_stops
print(f"Total stopwords: {len(ALL_STOPS)}")

Total stopwords: 457


Tokenizer for mixed-script text

In [ ]:
import re
from indicnlp.tokenize import indic_tokenize

def tokenize_mixed(text):
    """
    Tokenizes mixed Devanagari + Roman text.
    Devanagari chunks → indic tokenizer.
    Roman chunks → lowercase split.
    Filters stopwords, punctuation, single chars, digits.
    """
    tokens = []
    for word in text.split():
        word = word.strip("।॥.,!?;:\"'()[]{}—-")
        if not word or len(word) < 2:
            continue
        if re.search(r'[\u0900-\u097F]', word):
            # Devanagari — use indic tokenizer
            for t in indic_tokenize.trivial_tokenize(word, lang='hi'):
                t = t.strip()
                if t and len(t) > 1 and t not in ALL_STOPS and not t.isdigit():
                    tokens.append(t)
        else:
            # Roman — lowercase and filter
            w = word.lower()
            if w and len(w) > 1 and w not in ALL_STOPS and not w.isdigit() and w.replace("'","").isalnum():
                tokens.append(w)
    return tokens

Helper functions

In [ ]:
import numpy as np
np.random.seed(0)

def print_top_words(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(model.components_):
        top_idx   = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_idx]
        print(f"\n{topic_idx+1}: {', '.join(top_words)}")

def save_top_words(model, feature_names, n_top_words, out_path):
    with open(out_path, "w", encoding="utf-8-sig") as f:
        for topic_idx, topic in enumerate(model.components_):
            top_idx   = topic.argsort()[:-n_top_words - 1:-1]
            top_words = [feature_names[i] for i in top_idx]
            f.write(f"{topic_idx+1}," + ",".join(top_words) + "\n")
    print(f"  Saved topics → {out_path}")

LDA model per community

In [ ]:
import json, glob
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

for community, folder in COMMUNITY_FOLDERS.items():
    print(f"\n{'='*60}")
    print(f"Community: {community}")
    print(f"{'='*60}")

    # --- Load transcripts ---
    records = []
    for fpath in sorted(glob.glob(os.path.join(folder, "*.json"))):
        with open(fpath, encoding="utf-8") as f:
            data = json.load(f)
        if "error" in data:
            continue
        text = data.get("full_text_mixed", "").strip()
        if not text:
            continue
        channel = os.path.basename(fpath).split("__")[0]
        records.append({
            "filename": os.path.basename(fpath),
            "channel":  channel,
            "language": data.get("language", ""),
            "text":     text
        })

    df = pd.DataFrame(records)
    df["text"] = df["text"].fillna("")
    print(f"Loaded {len(df)} transcripts from {df['channel'].nunique()} channels")

    if len(df) < 10:
        print(f"  Too few transcripts — skipping {community}")
        continue

    # --- Tokenize ---
    print("  Tokenizing...")
    df["tokenized"] = df["text"].apply(
        lambda t: " ".join(tokenize_mixed(t))
    )
    df = df[df["tokenized"].str.strip() != ""].reset_index(drop=True)
    print(f"  Non-empty after tokenization: {len(df)}")

    # --- Document-term matrix ---
    vectorizer = CountVectorizer(
        token_pattern=r"(?u)\b\w+\b",  # Unicode-aware — handles Devanagari
        min_df=5,
        max_df=0.95
    )
    X = vectorizer.fit_transform(df["tokenized"])
    feature_names = vectorizer.get_feature_names_out()
    print(f"  Vocabulary size: {len(feature_names)} | Matrix: {X.shape}")

    # --- Train LDA ---
    print(f"  Fitting LDA ({N_TOPICS} topics)...")
    lda = LatentDirichletAllocation(
        n_components=N_TOPICS,
        learning_method="online",
        random_state=0,
        max_iter=20,
        evaluate_every=1,
        verbose=1
    )
    lda.fit(X)

    # --- Save top words ---
    topics_path = os.path.join(OUTPUT_DIR, f"{community}_LDA_topics.csv")
    save_top_words(lda, feature_names, N_TOP_WORDS, topics_path)

    # --- Document-topic distributions ---
    doc_topic_probs = lda.transform(X)
    topic_cols = pd.DataFrame(
        doc_topic_probs,
        columns=[f"Topic_{i+1}_Probability" for i in range(N_TOPICS)]
    )
    df["dominant_topic"] = doc_topic_probs.argmax(axis=1) + 1

    df_out = pd.concat(
        [df[["filename", "channel", "language", "dominant_topic"]].reset_index(drop=True),
         topic_cols],
        axis=1
    )

    # Drop any stray Unnamed columns
    df_out = df_out.loc[:, ~df_out.columns.str.contains("Unnamed")]

    # --- Save df ---
    df_path = os.path.join(OUTPUT_DIR, f"{community}_lda_features.csv")
    if os.path.exists(df_path):
        os.remove(df_path)
    df_out.to_csv(df_path, index=False, encoding="utf-8-sig")
    print(f"  Saved features → {df_path}")

print("\nAll communities done.")


Community: business_finance


KeyError: 'text'